<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook computes site-frequency spectra for the ~1,027-sample GRCh38 GLnexus joint-call Hail MatrixTable. It restricts to autosomal biallelic SNVs and indels, histograms alternate-allele frequency, and compares the spectra to a scaled 1/i expectation.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

# Compute folded site-frequency spectrum (GRCh38)

## Standard imports

In [ ]:
import os
import sys
import pandas as pd
import re

import math
import datetime
from dateutil import parser
from time import sleep
from typing import List
import pprint

from google.cloud import storage
my_storage_client = storage.Client()

from tqdm.notebook import tqdm

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import hail as hl
from hail.plot import output_notebook, show

from bokeh.resources import INLINE
import bokeh.io

bokeh.io.output_notebook(INLINE)

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']       # the workspace bucket
workspace = os.environ['WORKSPACE_NAME']      # workspace name
namespace = os.environ['WORKSPACE_NAMESPACE'] # workspace namespace

## Set paths

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Input: the 1,027-sample GLnexus joint-call MatrixTable. Outputs: the SNV and indel spectra, saved as text.
mt_path = f'{AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1027/cohort_for_GLNexus_2023Q1_1027.mt'

In [ ]:
snv_qc_sfs_path = f'{bucket}/scratch/snv_qc_sfs.txt'

In [ ]:
indel_qc_sfs_path = f'{bucket}/scratch/indel_qc_sfs.txt'

## Initialize Hail

In [ ]:
hl.init(default_reference='GRCh38')

## Compute site frequency spectrum

In [ ]:
# SNVs: autosomal, biallelic SNP sites.
mt = hl.read_matrix_table(mt_path)

In [ ]:
mt_snv = mt.filter_rows(
    (mt.locus.contig != 'chrX') &
    (mt.locus.contig != 'chrY') &
    (mt.locus.contig != 'chrM')
)
mt_snv.count()

In [ ]:
mt_snv = mt_snv.filter_rows(hl.len(mt_snv.alleles) == 2)
mt_snv.count()

In [ ]:
mt_snv = mt_snv.filter_rows(hl.is_snp(mt_snv.alleles[0], mt_snv.alleles[1]))
mt_snv.count()

In [ ]:
# Alt allele frequency per site (Hail variant_qc), histogrammed into 1,027 bins between 0 and 1.
snv_qc_rows = hl.variant_qc(mt_snv).rows()

In [ ]:
snv_qc_rows.variant_qc.show()

In [ ]:
snv_qc_sfs = snv_qc_rows.aggregate(hl.agg.hist(snv_qc_rows.variant_qc.AF[1], 0, 1, 1027))

In [ ]:
# Save the histogram, then read it back (so the plots can be redrawn without recomputing it).
with hl.hadoop_open(snv_qc_sfs_path, 'w') as f:
    f.write(str(dict(snv_qc_sfs)))

In [ ]:
snv_qc_sfs = hl.hadoop_open(snv_qc_sfs_path)
snv_sfs_dict = eval(snv_qc_sfs.read())
snv_sfs_struct = hl.Struct(**snv_sfs_dict)

In [ ]:
# Histogram on a log scale.
snv_sfs_p = hl.plot.histogram(snv_sfs_struct, log=True, legend="Frequency of SNV major allele at site")
show(snv_sfs_p)

In [ ]:
# Scratch checks; bins_data and freq_data are set in the next cell.
len(snv_sfs_struct.bin_edges)

In [ ]:
len(snv_sfs_struct.bin_freq)

In [ ]:
len(bins_data)

In [ ]:
len(freq_data)

In [ ]:
# The spectrum with a scaled 1/i curve, the shape expected for a neutral population of constant size.
bins_data = snv_sfs_struct.bin_edges
freq_data = snv_sfs_struct.bin_freq

plt.hist(bins_data[:-1], bins_data, weights=freq_data, log=True, alpha=0.5, label="Frequency of SNV major allele at site")

# Add 1/i curve
n = 1027
i_values = np.arange(1, n)
y_values = (1 / i_values)

scaling_factor = 0.105*max(freq_data)

plt.plot(i_values / (n - 1), y_values * scaling_factor, label='1/i curve', color='red')
plt.legend()
plt.xlabel("Allele Frequency")
plt.ylabel("Frequency")
plt.title("Site Frequency Spectrum with 1/i Curve")

plt.show()

In [ ]:
# Same for indels: autosomal, biallelic indel sites.
mt_indel = mt.filter_rows(
    (mt.locus.contig != 'chrX') &
    (mt.locus.contig != 'chrY') &
    (mt.locus.contig != 'chrM')
)
mt_indel.count()

In [ ]:
mt_indel = mt_indel.filter_rows(hl.len(mt_indel.alleles) == 2)
mt_indel.count()

In [ ]:
mt_indel = mt_indel.filter_rows(hl.is_indel(mt_indel.alleles[0], mt_indel.alleles[1]))
mt_indel.count()

In [ ]:
indel_qc_rows = hl.variant_qc(mt_indel).rows()

In [ ]:
indel_qc_rows.variant_qc.show()

In [ ]:
indel_qc_sfs = indel_qc_rows.aggregate(hl.agg.hist(indel_qc_rows.variant_qc.AF[1], 0, 1, 1027))

In [ ]:
with hl.hadoop_open(indel_qc_sfs_path, 'w') as f:
    f.write(str(dict(indel_qc_sfs)))

In [ ]:
indel_qc_sfs = hl.hadoop_open(indel_qc_sfs_path)
indel_sfs_dict = eval(indel_qc_sfs.read())
indel_sfs_struct = hl.Struct(**indel_sfs_dict)

In [ ]:
indel_sfs_p = hl.plot.histogram(indel_sfs_struct, log=True, legend="Frequency of indel major allele at site")
show(indel_sfs_p)

In [ ]:
bins_data = indel_sfs_struct.bin_edges
freq_data = indel_sfs_struct.bin_freq

plt.hist(bins_data[:-1], bins_data, weights=freq_data, log=True, alpha=0.5, label="Frequency of indel major allele at site")

# Add 1/i curve
n = 1027
i_values = np.arange(1, n)
y_values = (1 / i_values)

scaling_factor = 0.105*max(freq_data)

plt.plot(i_values / (n - 1), y_values * scaling_factor, label='1/i curve', color='red')

plt.legend()
plt.xlabel("Allele Frequency")
plt.ylabel("Frequency")
plt.title("Site Frequency Spectrum with 1/i Curve")

plt.show()